# Smart Home Gas-Leak — Train Forecaster + PPO trên Colab

Notebook tự chứa: chạy hết các cell theo thứ tự, cuối cùng tải về 2 file:
- `gas_forecaster.keras` — đặt vào `processing/ml/lstm/`
- `ppo_gas_agent.zip` — đặt vào `processing/ml/rl/`

**Khuyến nghị**: Runtime → Change runtime type → **GPU (T4)** để forecaster nhanh hơn.

## 1. Cài dependencies

In [ ]:
!pip install -q "tensorflow>=2.17,<2.20" "gymnasium==0.29.1" "stable-baselines3==2.3.2" "numpy>=1.26,<2.1"
import tensorflow as tf, stable_baselines3, gymnasium
print('TF', tf.__version__, '| SB3', stable_baselines3.__version__, '| Gym', gymnasium.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Ghi simulator (state machine vật lý)

In [ ]:
%%writefile sensor_simulator.py
from __future__ import annotations
import math, random
from dataclasses import dataclass
from enum import Enum

LEAK_PROB_PER_MIN = 0.05   # ~1 leak per 20 min when NORMAL
CRITICAL_PPM = 1000.0

class State(str, Enum):
    NORMAL = 'NORMAL'; LEAK_SLOW = 'LEAK_SLOW'; LEAK_FAST = 'LEAK_FAST'; VENTILATING = 'VENTILATING'

@dataclass
class World:
    gas: float = 60.0
    temp: float = 28.0
    hum: float = 60.0
    state: State = State.NORMAL
    state_age_s: float = 0.0

def _step_normal(w, dt):
    w.gas += (60.0 - w.gas) * 0.1 * dt + random.gauss(0, 3) * dt
    w.gas = max(20.0, min(w.gas, 200.0))

def _step_leak_slow(w, dt):
    w.gas += 5.0 * dt + random.gauss(0, 2) * dt
    w.gas = min(w.gas, 1500.0)

def _step_leak_fast(w, dt):
    w.gas = w.gas * math.exp(0.04 * dt) + 8.0 * dt + random.gauss(0, 3) * dt
    w.gas = min(w.gas, 2000.0)

def _step_ventilating(w, dt):
    w.gas = max(60.0, w.gas * math.exp(-0.03 * dt) - 0.5 * dt)

STEP_FN = {State.NORMAL: _step_normal, State.LEAK_SLOW: _step_leak_slow,
           State.LEAK_FAST: _step_leak_fast, State.VENTILATING: _step_ventilating}

def _maybe_transition(w, dt):
    p = LEAK_PROB_PER_MIN / 60.0 * dt
    if w.state == State.NORMAL and random.random() < p:
        w.state = State.LEAK_SLOW if random.random() < 0.7 else State.LEAK_FAST
        w.state_age_s = 0.0
        return
    if w.state in (State.LEAK_SLOW, State.LEAK_FAST):
        # Self-resolve faster so RL gets balanced episodes
        if w.state_age_s > 180 and random.random() < 0.005:
            w.state = State.VENTILATING
            w.state_age_s = 0.0
            return
    if w.state == State.VENTILATING and w.gas < 100 and w.state_age_s > 30:
        w.state = State.NORMAL
        w.state_age_s = 0.0

def _seconds_to_critical(w):
    if w.gas >= CRITICAL_PPM: return 0
    if w.state == State.LEAK_SLOW: return max(0, int((CRITICAL_PPM - w.gas) / 5.0))
    if w.state == State.LEAK_FAST and w.gas > 0: return max(0, int(math.log(CRITICAL_PPM / w.gas) / 0.04))
    return -1

## 3. Ghi Gymnasium environment cho RL

In [ ]:
%%writefile gas_env.py
from __future__ import annotations
import math
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from sensor_simulator import State, World, STEP_FN, _maybe_transition

ACTION_NAMES = ('NO_OP','ALERT_USER','FAN_ON','CLOSE_VALVE')
CRITICAL = 1000.0
HORIZON = 300

class GasLeakEnv(gym.Env):
    metadata = {'render_modes': []}
    def __init__(self, episode_seconds=1800, seed=None):
        super().__init__()
        self.episode_seconds = episode_seconds
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(8,), dtype=np.float32)
        self.action_space = spaces.Discrete(4)
        self.world = World(); self.t = 0
        self.fan_on = False; self.valve_closed = False
        self.time_since_action = 0; self._gas_history = []

    def reset(self, *, seed=None, options=None):
        if seed is not None:
            import random; random.seed(seed); np.random.seed(seed)
        self.world = World(); self.t = 0
        self.fan_on = False; self.valve_closed = False
        self.time_since_action = 0; self._gas_history = []
        return self._observe(0.0), {}

    def step(self, action):
        cost = 0.0
        leak = self.world.state
        if action == 1:
            cost = -2.0 if leak == State.NORMAL else 0.0
            self.time_since_action = 0
        elif action == 2:
            if not self.fan_on:
                self.fan_on = True
                cost = -5.0 if leak == State.NORMAL else 0.0
            self.time_since_action = 0
        elif action == 3:
            if not self.valve_closed:
                self.valve_closed = True
                if leak in (State.LEAK_SLOW, State.LEAK_FAST):
                    self.world.state = State.VENTILATING
                    self.world.state_age_s = 0.0
                else:
                    cost = -5.0
            self.time_since_action = 0
        else:
            self.time_since_action += 1

        STEP_FN[self.world.state](self.world, 1.0)
        if self.fan_on and self.world.state != State.VENTILATING:
            self.world.gas = max(50.0, self.world.gas * 0.985)
        _maybe_transition(self.world, 1.0)
        self.world.state_age_s += 1.0
        self.t += 1
        self._gas_history.append(self.world.gas)

        reward = -0.05 + cost
        if self.world.gas >= CRITICAL:
            reward -= 50.0
        if action in (2, 3) and leak in (State.LEAK_SLOW, State.LEAK_FAST):
            reward += 10.0

        truncated = self.t >= self.episode_seconds
        return self._observe(self._cheap_forecast()), float(reward), False, truncated, {
            'leak_state': self.world.state.value, 'gas_ppm': self.world.gas,
        }

    def _cheap_forecast(self):
        if len(self._gas_history) < 5: return 0.0
        n = min(30, len(self._gas_history))
        x = np.arange(n, dtype=np.float32)
        slope, _ = np.polyfit(x, self._gas_history[-n:], 1)
        predicted = self.world.gas + slope * HORIZON
        return float(1.0 / (1.0 + math.exp(-(predicted - CRITICAL) / 200.0)))

    def _slope(self):
        if len(self._gas_history) < 5: return 0.5
        n = min(30, len(self._gas_history))
        x = np.arange(n, dtype=np.float32)
        slope, _ = np.polyfit(x, self._gas_history[-n:], 1)
        return float(np.clip((slope + 10.0) / 20.0, 0.0, 1.0))

    def _observe(self, p_forecast):
        return np.array([
            min(self.world.gas / 2000.0, 1.0),
            min(self.world.temp / 60.0, 1.0),
            min(self.world.hum / 100.0, 1.0),
            self._slope(),
            float(np.clip(p_forecast, 0.0, 1.0)),
            1.0 if self.fan_on else 0.0,
            1.0 if self.valve_closed else 0.0,
            min(self.time_since_action / 300.0, 1.0),
        ], dtype=np.float32)

## 4. Train Forecasting LSTM
Sinh data từ simulator → train Keras → save `gas_forecaster.keras`.

In [ ]:
import random, numpy as np, tensorflow as tf
from sensor_simulator import World, STEP_FN, _maybe_transition

SEQ_LEN, HORIZON, CRITICAL = 60, 300, 1000.0
HOURS, EPOCHS, SEED = 8, 25, 42

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

n = int(HOURS * 3600)
trace = np.zeros((n, 4), dtype=np.float32)
w = World()
for t in range(n):
    STEP_FN[w.state](w, 1.0); _maybe_transition(w, 1.0); w.state_age_s += 1.0
    trace[t] = (w.gas, w.temp, w.hum, {'NORMAL':0,'LEAK_SLOW':1,'LEAK_FAST':2,'VENTILATING':3}[w.state.value])

print(f'trace shape={trace.shape}  leak frac={(trace[:,3]>0).mean():.2%}')

bounds = np.array([2000.0, 60.0, 100.0], dtype=np.float32)
feats = trace[:, :3] / bounds
gas = trace[:, 0]
last = n - HORIZON - 1
n_samples = last - SEQ_LEN
X = np.zeros((n_samples, SEQ_LEN, 3), dtype=np.float32)
y = np.zeros(n_samples, dtype=np.float32)
for i in range(n_samples):
    end = SEQ_LEN + i
    X[i] = feats[i:end]
    y[i] = 1.0 if gas[end+1:end+1+HORIZON].max() > CRITICAL else 0.0

pos = float(y.mean())
print(f'X={X.shape}  positive={pos:.2%}')

split = int(len(X) * 0.8)
Xtr, ytr, Xte, yte = X[:split], y[:split], X[split:], y[split:]

inp = tf.keras.layers.Input(shape=(SEQ_LEN, 3))
h = tf.keras.layers.LSTM(32)(inp)
h = tf.keras.layers.Dropout(0.2)(h)
h = tf.keras.layers.Dense(16, activation='relu')(h)
out = tf.keras.layers.Dense(1, activation='sigmoid')(h)
model = tf.keras.Model(inp, out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Precision(name='p'), tf.keras.metrics.Recall(name='r')])
model.summary()

history = model.fit(Xtr, ytr, validation_data=(Xte, yte), epochs=EPOCHS, batch_size=128,
                    class_weight={0: 1.0, 1: max(1.0, (1-pos)/max(pos,1e-3))}, verbose=2)
model.save('gas_forecaster.keras')
print('Saved gas_forecaster.keras')

## 5. Train PPO RL agent

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from gas_env import GasLeakEnv

STEPS = 500_000
env = make_vec_env(lambda: GasLeakEnv(episode_seconds=1800), n_envs=8, seed=42)
model = PPO('MlpPolicy', env, n_steps=512, batch_size=128, gae_lambda=0.95,
            gamma=0.99, learning_rate=3e-4, ent_coef=0.02, verbose=1, seed=42)
model.learn(total_timesteps=STEPS)
model.save('ppo_gas_agent.zip')
print('Saved ppo_gas_agent.zip')

## 6. Đánh giá nhanh: 3 controllers

In [ ]:
import random, math
import numpy as np
import tensorflow as tf
from stable_baselines3 import PPO
from sensor_simulator import World, STEP_FN, _maybe_transition, State

CRITICAL = 1000.0
forecaster = tf.keras.models.load_model('gas_forecaster.keras')
policy = PPO.load('ppo_gas_agent.zip')

def forecast_proba(buf):
    if len(buf) < 60: return 0.0
    arr = np.array(buf[-60:], dtype=np.float32)
    arr = arr / np.array([2000.0, 60.0, 100.0], dtype=np.float32)
    return float(forecaster.predict(arr.reshape(1, 60, 3), verbose=0)[0][0])

def run(controller, hours=2.0, seed=42):
    random.seed(seed); np.random.seed(seed)
    w = World()
    n = int(hours * 3600)
    leaks_total = 0; lead_times = []
    fa = 0; normal_s = 0; peaks = []
    in_leak = False; leak_peak = 0; first_alarm_dt = None; leak_started = -1
    fan_on = False; valve_closed = False
    buf = []
    gas_hist = []

    for t in range(n):
        STEP_FN[w.state](w, 1.0)
        if fan_on and w.state != State.VENTILATING:
            w.gas = max(50.0, w.gas * 0.985)
        _maybe_transition(w, 1.0); w.state_age_s += 1.0
        buf.append([w.gas, w.temp, w.hum])
        gas_hist.append(w.gas)

        leaking = w.state in (State.LEAK_SLOW, State.LEAK_FAST)
        if leaking and not in_leak:
            in_leak = True; leaks_total += 1; leak_started = t
            first_alarm_dt = None; leak_peak = w.gas
        elif in_leak:
            leak_peak = max(leak_peak, w.gas)
            if not leaking:
                peaks.append(leak_peak); in_leak = False
                fan_on = False; valve_closed = False
        else:
            normal_s += 1

        # Predict every 5s only (TF speed)
        p5 = forecast_proba(buf) if t % 5 == 0 else 0.0

        if controller == 'threshold':
            action = 1 if w.gas > 800 else 0
        elif controller == 'forecaster':
            action = 1 if p5 > 0.5 else 0
        else:  # rl
            slope = 0.5
            if len(gas_hist) >= 5:
                m = min(30, len(gas_hist))
                x = np.arange(m, dtype=np.float32)
                s, _ = np.polyfit(x, gas_hist[-m:], 1)
                slope = float(np.clip((s + 10) / 20, 0, 1))
            obs = np.array([min(w.gas/2000,1), min(w.temp/60,1), min(w.hum/100,1),
                            slope, p5, 1.0 if fan_on else 0.0,
                            1.0 if valve_closed else 0.0, 0.0], dtype=np.float32)
            a, _ = policy.predict(obs, deterministic=True)
            action = int(a)

        if action == 0: continue
        if not in_leak: fa += 1
        elif first_alarm_dt is None:
            # Walk forward to find when gas would have hit critical
            future_t = leak_started + min(600, max(60, int((CRITICAL - w.gas) / max(0.5, w.gas - 60) * 60)))
            first_alarm_dt = max(0, future_t - t)
            lead_times.append(first_alarm_dt)
        if action == 2: fan_on = True
        elif action == 3 and in_leak:
            valve_closed = True
            w.state = State.VENTILATING; w.state_age_s = 0.0

    return {
        'controller': controller,
        'leaks': leaks_total,
        'lead_s': float(np.mean(lead_times)) if lead_times else 0.0,
        'fa_per_min': fa / max(1, normal_s/60.0),
        'mean_peak': float(np.mean(peaks)) if peaks else 0.0,
    }

rows = [run(c, hours=2.0, seed=42) for c in ('threshold','forecaster','rl')]
print(f"\n{'controller':<12} {'leaks':>6} {'lead_s':>8} {'fa/min':>8} {'peak_gas':>10}")
print('-'*50)
for r in rows:
    print(f"{r['controller']:<12} {r['leaks']:>6} {r['lead_s']:>8.1f} {r['fa_per_min']:>8.2f} {r['mean_peak']:>10.1f}")

## 7. Tải models về máy

In [ ]:
from google.colab import files
files.download('gas_forecaster.keras')
files.download('ppo_gas_agent.zip')

## 8. Đặt models vào dự án

Sau khi 2 file tải về máy, copy chúng vào:

```
thesis-smart-home-gas-detection/processing/ml/lstm/gas_forecaster.keras
thesis-smart-home-gas-detection/processing/ml/rl/ppo_gas_agent.zip
```

Sau đó: `docker compose up --build` — pipeline sẽ tự load models và dashboard hiển thị `Predicted Risk in 5 min` + `Auto Action (RL)`.